# Synthetic Saudi Diabetes Population Generator

**Purpose:** Generate a synthetic Saudi population dataset for educational and statistical analysis.

- Sample size: 10,000 synthetic records
- Population reference: Saudi nationals aged 15+
- Reference year: 2022 population structure
- Scenarios: Low, Central, High

**Important:** These are synthetic records, not real patients. Published statistics are used as reference targets, with assumptions documented separately.

## Imports & Configuration

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.optimize import brentq
from scipy.special import expit
from scipy.stats import truncnorm

SEED = 42
N = 10_000

print("Libraries imported successfully.")
print(f"Sample size: {N:,}")

Libraries imported successfully.
Sample size: 10,000


## Population Structure

In [2]:
# 1. POPULATION: Saudi nationals, census 2022 (GAStat), age 15+

BANDS = [
    (15, 19), (20, 24), (25, 29), (30, 34), (35, 39),
    (40, 44), (45, 49), (50, 54), (55, 59), (60, 64),
    (65, 85)
]

POP_2022 = {
    "M": [
        941_806, 868_712, 859_531, 770_595, 679_028,
        534_889, 381_557, 310_703, 252_068, 186_840,
        324_279
    ],
    "F": [
        916_439, 850_780, 842_731, 758_562, 675_023,
        543_104, 404_356, 343_291, 276_936, 203_802,
        335_592
    ],
}

POP_SAUDI_15PLUS = sum(sum(v) for v in POP_2022.values())

REPORT_GROUPS = [
    "15-24", "25-34", "35-44", "45-54", "55-64", "65+"
]

REPORT_BINS = [14, 24, 34, 44, 54, 64, 120]


print("Population parameters loaded.")

Population parameters loaded.


## Published Target & Sizes

In [3]:
import numpy as np

def LN(x):
    return np.log(x)

In [4]:
CASCADE = {"undiagnosed": 0.578, "treated_uncontrolled": 0.202,    # SHIS 2013 (VERIFIED)
           "treated_controlled": 0.166, "untreated": 0.054}
BMI_CATS = ["underweight", "normal", "overweight", "obese", "morbid"]
BMI_SHIS = {"M": [0.074, 0.356, 0.331, 0.213, 0.026],              # Memish 2014 (VERIFIED)
            "F": [0.064, 0.327, 0.278, 0.284, 0.047]}
OBESITY_TARGET = {"M": 0.241, "F": 0.335}                          # BMI>=30 (VERIFIED)
GDM_RANGE = (0.13, 0.176)                                          # MENA 13.0%, Saudi 17.6%
BMI_AGE_TILT = [-0.60, -0.10, 0.15, 0.35, 0.35]                    # direction verified, size ASSUMPTION
BMI_RANGES = {"underweight": (15, 18.5), "normal": (18.5, 25),
              "overweight": (25, 30), "obese": (30, 40)}           # morbid = 40 + exp, cap 60

AGE_OR_TARGET = 4.47          # crude OR age>=40 vs <40, Bisha 2017 (regional, WEAK)
B = {
    "male": 0.20,             # ASSUMPTION
    "family_hx": 0.50,        # ASSUMPTION
    "bmi": {"underweight": 0.0, "normal": 0.0, "overweight": LN(3.20),   # Jeddah 2019 (weak)
            "obese": LN(3.20) + 0.30, "morbid": LN(3.20) + 0.60},        # increments ASSUMPTION
    "pre_overweight_plus": LN(2.50),                                     # Jeddah 2019 (weak)
    "pre_age_per_decade": 0.30,                                          # ASSUMPTION
}
P_FAMILY_HX = 0.35                           # ASSUMPTION
P_T1_UNDER25, P_T1_25PLUS = 0.10, 0.01       # ASSUMPTION
P_PREGNANT_15_49 = 0.06                      # ASSUMPTION (GDM module only)

## Cumulative sugar & associated diseases

In [5]:
# (low, high, mean, sd) truncated normals - ASSUMPTIONS consistent with diagnostic cut-offs
HBA1C_BY_STATUS = {"normal": (4.5, 5.69, 5.3, 0.30),
                   "prediabetes": (5.7, 6.49, 6.0, 0.20)}
HBA1C_BY_CASCADE = {"undiagnosed": (6.5, 12, 7.5, 1.2),
                    "untreated": (6.5, 12, 7.8, 1.3),
                    "treated_controlled": (6.0, 6.99, 6.5, 0.3),
                    "treated_uncontrolled": (7.0, 12, 8.5, 1.3)}

# NOT calibrated. HTN OR 5.08 verified (claims data); the rest are ASSUMPTIONS
COMORB = {
    "hypertension":         {"b0": -2.2, "age": 0.45, "obese": 0.5, "diabetes": LN(5.08)},
    "hypercholesterolemia": {"b0": -2.4, "age": 0.35, "obese": 0.4, "diabetes": 0.8},
}

Help Functions

In [6]:
def tn(rng, lo, hi, loc, sd, size):
    a, b = (lo - loc) / sd, (hi - loc) / sd
    return truncnorm.rvs(
        a, b, loc=loc, scale=sd, size=size, random_state=rng
    )


def sample_population(rng, n):
    cells, p = [], []

    for s in ("M", "F"):
        for band, cnt in zip(BANDS, POP_2022[s]):
            cells.append((s, band))
            p.append(cnt)

    p = np.array(p) / sum(p)

    pick = rng.choice(len(cells), n, p=p)

    sex = np.array([cells[i][0] for i in pick])

    age = np.array([
        rng.integers(
            cells[i][1][0],
            cells[i][1][1] + 1
        )
        for i in pick
    ])

    return sex, age


def bmi_categories(rng, sex, age):
    """Age-tilted BMI category, with per-sex marginals re-fitted to SHIS 2013."""

    z = (np.minimum(age, 60) - 40) / 15
    cat = np.empty(len(age), dtype=object)

    for s in ("M", "F"):
        idx = sex == s
        target = np.array(BMI_SHIS[s])
        base = target.copy()

        for _ in range(40):
            w = base[None, :] * np.exp(
                np.outer(z[idx], BMI_AGE_TILT)
            )
            w /= w.sum(1, keepdims=True)
            base *= target / w.mean(0)

        w = base[None, :] * np.exp(
            np.outer(z[idx], BMI_AGE_TILT)
        )
        w /= w.sum(1, keepdims=True)

        u = rng.random(idx.sum())[:, None]

        cat[idx] = np.array(BMI_CATS)[
            (u > np.cumsum(w, 1)).sum(1).clip(0, 4)
        ]

    return cat


def solve_intercept(lp, target):
    return brentq(
        lambda t: expit(t + lp).mean() - target,
        -25,
        5
    )


def expected_or_ge40(p, age):
    hi = age >= 40

    a, b = p[hi].sum(), (1 - p[hi]).sum()
    c, d = p[~hi].sum(), (1 - p[~hi]).sum()

    return (a * d) / (b * c)


def calibrate_age_slope(
    lp_other, age, d_prev, target_or=AGE_OR_TARGET
):
    """Calibrate the age slope to a target crude OR."""

    def f(sl):
        lp = lp_other + sl * (age - 45) / 10
        t = solve_intercept(lp, d_prev)

        return (
            np.log(expected_or_ge40(expit(t + lp), age))
            - np.log(target_or)
        )

    return brentq(f, 0.01, 4.0)

In [7]:
# Helper function: Truncated Normal Distribution

def _truncnorm(rng, low, high, mean, sd, size):
    size = int(size)

    if size == 0:
        return np.empty(0, dtype=float)

    if sd <= 0 or low >= high:
        raise ValueError("Invalid truncated normal parameters.")

    values = rng.normal(loc=mean, scale=sd, size=size)
    invalid = (values < low) | (values > high)
    attempts = 0

    while np.any(invalid) and attempts < 100:
        values[invalid] = rng.normal(
            loc=mean,
            scale=sd,
            size=int(invalid.sum())
        )
        invalid = (values < low) | (values > high)
        attempts += 1

    return np.clip(values, low, high)

In [8]:
print("HBA1C_BY_STATUS:")
for name, params in HBA1C_BY_STATUS.items():
    print(name, repr(params))

print("\nHBA1C_BY_CASCADE:")
for name, params in HBA1C_BY_CASCADE.items():
    print(name, repr(params))

HBA1C_BY_STATUS:
normal (4.5, 5.69, 5.3, 0.3)
prediabetes (5.7, 6.49, 6.0, 0.2)

HBA1C_BY_CASCADE:
undiagnosed (6.5, 12, 7.5, 1.2)
untreated (6.5, 12, 7.8, 1.3)
treated_controlled (6.0, 6.99, 6.5, 0.3)
treated_uncontrolled (7.0, 12, 8.5, 1.3)


In [9]:
print("HBA1C_BY_STATUS:")
for name, params in HBA1C_BY_STATUS.items():
    print(name, repr(params), "length =", len(params))

print("\nHBA1C_BY_CASCADE:")
for name, params in HBA1C_BY_CASCADE.items():
    print(name, repr(params), "length =", len(params))

HBA1C_BY_STATUS:
normal (4.5, 5.69, 5.3, 0.3) length = 4
prediabetes (5.7, 6.49, 6.0, 0.2) length = 4

HBA1C_BY_CASCADE:
undiagnosed (6.5, 12, 7.5, 1.2) length = 4
untreated (6.5, 12, 7.8, 1.3) length = 4
treated_controlled (6.0, 6.99, 6.5, 0.3) length = 4
treated_uncontrolled (7.0, 12, 8.5, 1.3) length = 4


 ## Data Generation: Population, BMI & Diabetes

In [10]:
def sample_population(rng, n):
    cells_, weights = [], []
    for sex in ("M", "F"):
        for band, count in zip(BANDS, POP_2022[sex]):
            cells_.append((sex, band)); weights.append(count)
    p = np.array(weights) / sum(weights)
    pick = rng.choice(len(cells_), n, p=p)
    sex = np.array([cells_[i][0] for i in pick])
    age = np.array([rng.integers(cells_[i][1][0], cells_[i][1][1] + 1) for i in pick])
    return sex, age


def sample_bmi_category(rng, sex, age):
    """Age-tilted category; per-sex marginals re-fitted to SHIS 2013 (IPF)."""
    z = (np.minimum(age, 60) - 40) / 15
    cat = np.empty(len(age), dtype=object)
    for s in ("M", "F"):
        idx = sex == s
        target = np.array(BMI_SHIS[s])
        base = target.copy()
        for _ in range(40):
            w = base[None, :] * np.exp(np.outer(z[idx], BMI_AGE_TILT))
            w /= w.sum(1, keepdims=True)
            base *= target / w.mean(0)
        w = base[None, :] * np.exp(np.outer(z[idx], BMI_AGE_TILT))
        w /= w.sum(1, keepdims=True)
        u = rng.random(idx.sum())[:, None]
        cat[idx] = np.array(BMI_CATS)[(u > np.cumsum(w, 1)).sum(1).clip(0, 4)]
    return cat


def sample_bmi_value(rng, cat):
    bmi = np.empty(len(cat))
    for c in BMI_CATS:
        m = cat == c
        if c == "morbid":
            bmi[m] = np.minimum(40 + rng.exponential(3, m.sum()), 60)
        else:
            bmi[m] = rng.uniform(*BMI_RANGES[c], m.sum())
    return bmi

## Simulate The Sugar State

In [11]:
def simulate_diabetes(rng, age, male, family_hx, cat, prevalence):
    lp_other = (B["male"] * male + B["family_hx"] * family_hx
                + np.array([B["bmi"][c] for c in cat]))
    slope = calibrate_age_slope(lp_other, age, prevalence)
    lp = lp_other + slope * (age - 45) / 10
    intercept = solve_intercept(lp, prevalence)
    return rng.binomial(1, expit(intercept + lp)), slope


def simulate_prediabetes(rng, diabetic, age, ow_plus, share_of_all):
    nd = diabetic == 0
    lp = B["pre_age_per_decade"] * (age - 45) / 10 + B["pre_overweight_plus"] * ow_plus
    intercept = brentq(lambda t: expit(t + lp[nd]).sum() / len(age) - share_of_all, -20, 5)
    pre = np.zeros(len(age), dtype=int)
    pre[nd] = rng.binomial(1, expit(intercept + lp[nd]))
    return pre


def simulate_diabetes_type(rng, diabetic, age):
    p_t1 = np.where(age < 25, P_T1_UNDER25, P_T1_25PLUS)
    return np.where(diabetic == 1,
                    np.where(rng.random(len(age)) < p_t1, "T1", "T2"), "none")


def simulate_gdm(rng, sex, age, diabetic):
    """EXPERIMENTAL, not evidence-based."""
    pregnant = ((sex == "F") & (age <= 49)
                & (rng.random(len(age)) < P_PREGNANT_15_49)).astype(int)
    gdm = ((pregnant == 1) & (diabetic == 0)
           & (rng.random(len(age)) < np.mean(GDM_RANGE))).astype(int)
    return {"pregnant": pregnant, "gdm": gdm}


def simulate_cascade(rng, diabetic):
    cascade = np.full(len(diabetic), "none", dtype=object)
    di = diabetic == 1
    cascade[di] = rng.choice(list(CASCADE), di.sum(), p=list(CASCADE.values()))
    return cascade


def simulate_hba1c(rng, status, cascade):
    """Derived from status/cascade, so NOT independent of the label (leakage risk for ML)."""
    hba1c = np.empty(len(status))
    for name, params in HBA1C_BY_STATUS.items():
        m = status == name
        hba1c[m] = _truncnorm(rng, *params, int(m.sum()))
    for name, params in HBA1C_BY_CASCADE.items():
        m = cascade == name
        hba1c[m] = _truncnorm(rng, *params, int(m.sum()))
    return hba1c


def simulate_fasting_glucose(rng, hba1c):
    return np.clip(1.6 * hba1c - 3.2 + rng.normal(0, 0.6, len(hba1c)), 3.3, 25)


def simulate_comorbidities(rng, age, obese, diabetic):
    """Generated AFTER diabetes, so they are outcomes here, not predictors."""
    out = {}
    for name, c in COMORB.items():
        z = c["b0"] + c["age"] * (age - 45) / 10 + c["obese"] * obese + c["diabetes"] * diabetic
        out[name] = rng.binomial(1, expit(z))
    return out

## Synthetic Data Generation

In [12]:
# Scenario definitions
# PROVISIONAL VALUES — verify against the original study sources.

SCENARIOS = {
    "low": (
        0.15, "PROVISIONAL — verify source",
        0.10, "PROVISIONAL — verify source"
    ),
    "central": (
        0.18, "PROVISIONAL — verify source",
        0.12, "PROVISIONAL — verify source"
    ),
    "high": (
        0.22, "PROVISIONAL — verify source",
        0.15, "PROVISIONAL — verify source"
    ),
}

SCENARIO_LABELS = {
    "low": "Low prevalence scenario",
    "central": "Central prevalence scenario",
    "high": "High prevalence scenario",
}

# Validate the configuration
assert set(SCENARIOS) == {"low", "central", "high"}

for name, (diabetes_prev, diabetes_src, prediabetes_share, prediabetes_src) in SCENARIOS.items():
    assert 0 < diabetes_prev < 1, f"Invalid diabetes prevalence: {name}"
    assert 0 <= prediabetes_share < 1, f"Invalid prediabetes share: {name}"

print("SCENARIOS and SCENARIO_LABELS defined successfully.")

SCENARIOS and SCENARIO_LABELS defined successfully.


In [13]:
# Enable gestational diabetes simulation
ENABLE_GDM = True

In [14]:
def generate(scenario="central", n=N, seed=SEED):
    """Returns SYNTHETIC records. df.attrs['age_slope'] holds the solved age slope."""
    rng = np.random.default_rng(seed)
    prevalence, _, pre_share, _ = SCENARIOS[scenario]

    sex, age = sample_population(rng, n)
    male = (sex == "M").astype(int)
    age_group = pd.cut(age, REPORT_BINS, labels=REPORT_GROUPS).astype(str)

    cat = sample_bmi_category(rng, sex, age)
    bmi = sample_bmi_value(rng, cat)
    obese = np.isin(cat, ["obese", "morbid"]).astype(int)
    ow_plus = np.isin(cat, ["overweight", "obese", "morbid"]).astype(int)
    family_hx = rng.binomial(1, P_FAMILY_HX, n)

    diabetic, slope = simulate_diabetes(rng, age, male, family_hx, cat, prevalence)
    pre = simulate_prediabetes(rng, diabetic, age, ow_plus, pre_share)
    status = np.where(diabetic == 1, "diabetes", np.where(pre == 1, "prediabetes", "normal"))

    dtype = simulate_diabetes_type(rng, diabetic, age)
    extra = simulate_gdm(rng, sex, age, diabetic) if ENABLE_GDM else {}
    cascade = simulate_cascade(rng, diabetic)
    hba1c = simulate_hba1c(rng, status, cascade)
    fpg = simulate_fasting_glucose(rng, hba1c)
    comorb = simulate_comorbidities(rng, age, obese, diabetic)

    df = pd.DataFrame({
        "id": [f"SYN{i:06d}" for i in range(n)], "sex": sex, "age": age,
        "age_group": age_group, "bmi": bmi.round(1), "bmi_category": cat,
        "family_history": family_hx, "diabetes_status": status, "diabetes_type": dtype,
        "care_cascade": cascade, "hba1c": hba1c.round(2),
        "fasting_glucose_mmol": fpg.round(1), **comorb, **extra,
        "scenario": scenario, "scenario_label": SCENARIO_LABELS[scenario],
        "synthetic": True})
    df.attrs["age_slope"] = slope
    return df

In [15]:
# Inspect existing calibration settings

settings = [
    "AGE_OR_TARGET",
    "P",
    "B",
    "CASCADE",
    "OBESITY_TARGET",
    "SCENARIO_LABELS",
    "PREDIABETES_TARGET",
    "PREVALENCE_TARGET",
]

for name in settings:
    if name in globals():
        print(f"\n{name} =")
        print(globals()[name])
    else:
        print(f"{name}: MISSING")


AGE_OR_TARGET =
4.47
P: MISSING

B =
{'male': 0.2, 'family_hx': 0.5, 'bmi': {'underweight': 0.0, 'normal': 0.0, 'overweight': np.float64(1.1631508098056809), 'obese': np.float64(1.463150809805681), 'morbid': np.float64(1.7631508098056807)}, 'pre_overweight_plus': np.float64(0.9162907318741551), 'pre_age_per_decade': 0.3}

CASCADE =
{'undiagnosed': 0.578, 'treated_uncontrolled': 0.202, 'treated_controlled': 0.166, 'untreated': 0.054}

OBESITY_TARGET =
{'M': 0.241, 'F': 0.335}

SCENARIO_LABELS =
{'low': 'Low prevalence scenario', 'central': 'Central prevalence scenario', 'high': 'High prevalence scenario'}
PREDIABETES_TARGET: MISSING
PREVALENCE_TARGET: MISSING


In [16]:
# Inspect variables used for prevalence calibration

for name in sorted(globals()):
    if any(term in name.lower() for term in [
        "preval", "prediabet", "scenario", "calibrat", "target"
    ]):
        if not name.startswith("_"):
            try:
                value = repr(globals()[name])
                print(f"{name} = {value[:500]}")
            except Exception:
                pass

AGE_OR_TARGET = 4.47
OBESITY_TARGET = {'M': 0.241, 'F': 0.335}
SCENARIOS = {'low': (0.15, 'PROVISIONAL — verify source', 0.1, 'PROVISIONAL — verify source'), 'central': (0.18, 'PROVISIONAL — verify source', 0.12, 'PROVISIONAL — verify source'), 'high': (0.22, 'PROVISIONAL — verify source', 0.15, 'PROVISIONAL — verify source')}
SCENARIO_LABELS = {'low': 'Low prevalence scenario', 'central': 'Central prevalence scenario', 'high': 'High prevalence scenario'}
calibrate_age_slope = <function calibrate_age_slope at 0x000002247D5ADFE0>
prediabetes_share = 0.15
prediabetes_src = 'PROVISIONAL — verify source'
simulate_prediabetes = <function simulate_prediabetes at 0x000002247D5AE560>


In [17]:
# Check scenario definitions and related settings

print("SCENARIOS:", globals().get("SCENARIOS", "MISSING"))
print("SCENARIO_LABELS:", globals().get("SCENARIO_LABELS", "MISSING"))
print("AGE_OR_TARGET:", globals().get("AGE_OR_TARGET", "MISSING"))
print("OBESITY_TARGET:", globals().get("OBESITY_TARGET", "MISSING"))
print("N:", globals().get("N", "MISSING"))
print("SEED:", globals().get("SEED", "MISSING"))

SCENARIOS: {'low': (0.15, 'PROVISIONAL — verify source', 0.1, 'PROVISIONAL — verify source'), 'central': (0.18, 'PROVISIONAL — verify source', 0.12, 'PROVISIONAL — verify source'), 'high': (0.22, 'PROVISIONAL — verify source', 0.15, 'PROVISIONAL — verify source')}
SCENARIO_LABELS: {'low': 'Low prevalence scenario', 'central': 'Central prevalence scenario', 'high': 'High prevalence scenario'}
AGE_OR_TARGET: 4.47
OBESITY_TARGET: {'M': 0.241, 'F': 0.335}
N: 10000
SEED: 42


## Titration & Examinations

In [18]:
def calibration_report(df, scenario="central"):
    """Tolerance = 3 SE of sampling noise (consistency, not proof of realism)."""
    d_prev, d_src, p_pre, p_src = SCENARIOS[scenario]
    rows = []

    def add(name, ref, sim, base_n, src):
        se = np.sqrt(ref * (1 - ref) / base_n)
        diff = sim - ref
        rows.append([name, ref, round(sim, 4), round(diff, 4),
                     round(diff / ref, 3), abs(diff) <= 3 * se, src])

    add("diabetes prevalence", d_prev, (df.diabetes_status == "diabetes").mean(), len(df), d_src)
    add("prediabetes share", p_pre, (df.diabetes_status == "prediabetes").mean(), len(df), p_src)
    dd = df[df.diabetes_status == "diabetes"]
    for k, v in CASCADE.items():
        add(f"cascade: {k}", v, (dd.care_cascade == k).mean(), len(dd), "SHIS 2013")
    for s in ("M", "F"):
        sub = df[df.sex == s]
        add(f"obesity {s}", OBESITY_TARGET[s], sub.bmi.ge(30).mean(), len(sub), "Memish 2014")
    return pd.DataFrame(rows, columns=["target", "published", "synthetic", "abs_diff",
                                       "rel_diff", "within_3SE", "source"])


def crude_or(a, b):
    t = pd.crosstab(a, b).reindex(index=[0, 1], columns=[0, 1], fill_value=0) + 0.5
    return (t.loc[1, 1] * t.loc[0, 0]) / (t.loc[1, 0] * t.loc[0, 1])


def joint_checks(df):
    dm = (df.diabetes_status == "diabetes").astype(int)
    print("== Joint-relationship checks (crude ORs) ==")
    print(f"HTN vs diabetes OR      : {crude_or(df.hypertension, dm):.2f}   (input 5.08)")
    print(f"BMI>=25 vs diabetes OR  : {crude_or((df.bmi >= 25).astype(int), dm):.2f}   (input 3.20; age confounding)")
    print(f"Age>=40 vs diabetes OR  : {crude_or((df.age >= 40).astype(int), dm):.2f}   (target {AGE_OR_TARGET})")
    print(f"Diabetes prevalence age>=50: {dm[df.age >= 50].mean():.1%}  (soft check only)")

## The Operation

In [19]:
SCENARIO = "central"            # "low" | "central" | "high"

df = generate(SCENARIO)
print(SCENARIO_LABELS[SCENARIO], f"| N = {len(df):,}")
print(f"Calibrated age slope: {df.attrs['age_slope']:.3f} log-odds/decade")
print("NOTE: ages inside 65+ are uniform 65-85; interpret 65+ as one group only.\n")

display(calibration_report(df, SCENARIO))      # use print(...) outside Jupyter
joint_checks(df)

print("\nStatus by age group (row %):")
display((pd.crosstab(df.age_group, df.diabetes_status, normalize="index") * 100)
        .reindex(REPORT_GROUPS).round(1))

df.to_csv(f"synthetic_saudi_diabetes_{SCENARIO}.csv", index=False)
print("Saved SYNTHETIC records. Not real patients.")

Central prevalence scenario | N = 10,000
Calibrated age slope: 0.526 log-odds/decade
NOTE: ages inside 65+ are uniform 65-85; interpret 65+ as one group only.



,target,published,synthetic,abs_diff,rel_diff,within_3SE,source
0,diabetes prevalence,0.180,0.1759,-0.0041,-0.023,True,PROVISIONAL — verify source
1,prediabetes share,0.120,0.1168,-0.0032,-0.027,True,PROVISIONAL — verify source
2,cascade: undiagnosed,0.578,0.5748,-0.0032,-0.006,True,SHIS 2013
3,cascade: treated_uncontrolled,0.202,0.2109,0.0089,0.044,True,SHIS 2013
4,cascade: treated_controlled,0.166,0.1603,-0.0057,-0.034,True,SHIS 2013
5,cascade: untreated,0.054,0.0540,0.0000,0.000,True,SHIS 2013
6,obesity M,0.241,0.2559,0.0149,0.062,True,Memish 2014
7,obesity F,0.335,0.3339,-0.0011,-0.003,True,Memish 2014


== Joint-relationship checks (crude ORs) ==
HTN vs diabetes OR      : 9.48   (input 5.08)
BMI>=25 vs diabetes OR  : 4.54   (input 3.20; age confounding)
Age>=40 vs diabetes OR  : 4.63   (target 4.47)
Diabetes prevalence age>=50: 43.1%  (soft check only)

Status by age group (row %):


diabetes_status,diabetes,normal,prediabetes
age_group,,,
15-24,6.2,84.9,9.0
25-34,10.9,78.6,10.5
35-44,17.9,68.3,13.8
45-54,29.5,56.2,14.2
55-64,39.6,45.0,15.4
65+,57.5,28.5,14.0


Saved SYNTHETIC records. Not real patients.


## Generate, Validate & Save 

In [20]:
from pathlib import Path
import numpy as np
import pandas as pd

# ── 1. Generate ──────────────────────────────────────────────

SCENARIO = "central"  # "low" | "central" | "high"

df = generate(SCENARIO)

print("=" * 55)
print("SYNTHETIC SAUDI DIABETES DATA")
print("=" * 55)
print(f"Scenario: {SCENARIO_LABELS[SCENARIO]}")
print(f"Records:  {len(df):,}")
print(f"Columns:  {df.shape[1]}")
print(f"Age slope: {df.attrs['age_slope']:.3f} log-odds/decade")
print("Note: Ages within 65+ are sampled from 65–85.")

# ── 2. Validate ──────────────────────────────────────────────

print("\n" + "=" * 55)
print("DATA QUALITY CHECKS")
print("=" * 55)

checks = {
    "Expected number of records": len(df) == N,
    "Unique record IDs": df["id"].is_unique,
    "No missing values": not df.isna().any().any(),
    "All records marked synthetic": df["synthetic"].eq(True).all(),
    "Valid diabetes status": df["diabetes_status"].isin(
        ["normal", "prediabetes", "diabetes"]
    ).all(),
    "Valid age range": df["age"].between(15, 85).all(),
    "Valid BMI values": df["bmi"].gt(0).all(),
    "Valid scenario labels": df["scenario"].eq(SCENARIO).all(),
}

for name, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'} | {name}")

quality_passed = all(checks.values())

print("\nDuplicate IDs:", df["id"].duplicated().sum())

print("\nDiabetes status distribution (%):")
display(
    (df["diabetes_status"].value_counts(normalize=True) * 100)
    .round(2)
    .to_frame("percentage")
)

print("\nStatus by age group (row %):")
display(
    (
        pd.crosstab(
            df["age_group"],
            df["diabetes_status"],
            normalize="index"
        ) * 100
    ).reindex(REPORT_GROUPS).round(1)
)

print("\nCalibration report:")
calibration = calibration_report(df, SCENARIO)
display(calibration)

print("\nJoint checks:")
joint_checks(df)

# ── 3. Save ──────────────────────────────────────────────────

if quality_passed:
    output_dir = Path("output")
    output_dir.mkdir(parents=True, exist_ok=True)

    output_file = output_dir / (
        f"synthetic_saudi_diabetes_{SCENARIO}.csv"
    )

    df.to_csv(output_file, index=False)

    print("\n" + "=" * 55)
    print("DATA SAVED SUCCESSFULLY")
    print("=" * 55)
    print(f"File: {output_file.resolve()}")
    print(f"Size: {output_file.stat().st_size / 1024:.2f} KB")
    print("These records are synthetic, not real patients.")
else:
    print("\nSAVE BLOCKED: Fix failed quality checks before saving.")

SYNTHETIC SAUDI DIABETES DATA
Scenario: Central prevalence scenario
Records:  10,000
Columns:  19
Age slope: 0.526 log-odds/decade
Note: Ages within 65+ are sampled from 65–85.

DATA QUALITY CHECKS
PASS | Expected number of records
PASS | Unique record IDs
PASS | No missing values
PASS | All records marked synthetic
PASS | Valid diabetes status
PASS | Valid age range
PASS | Valid BMI values
PASS | Valid scenario labels

Duplicate IDs: 0

Diabetes status distribution (%):


,percentage
diabetes_status,
normal,70.73
diabetes,17.59
prediabetes,11.68



Status by age group (row %):


diabetes_status,diabetes,normal,prediabetes
age_group,,,
15-24,6.2,84.9,9.0
25-34,10.9,78.6,10.5
35-44,17.9,68.3,13.8
45-54,29.5,56.2,14.2
55-64,39.6,45.0,15.4
65+,57.5,28.5,14.0



Calibration report:


,target,published,synthetic,abs_diff,rel_diff,within_3SE,source
0,diabetes prevalence,0.180,0.1759,-0.0041,-0.023,True,PROVISIONAL — verify source
1,prediabetes share,0.120,0.1168,-0.0032,-0.027,True,PROVISIONAL — verify source
2,cascade: undiagnosed,0.578,0.5748,-0.0032,-0.006,True,SHIS 2013
3,cascade: treated_uncontrolled,0.202,0.2109,0.0089,0.044,True,SHIS 2013
4,cascade: treated_controlled,0.166,0.1603,-0.0057,-0.034,True,SHIS 2013
5,cascade: untreated,0.054,0.0540,0.0000,0.000,True,SHIS 2013
6,obesity M,0.241,0.2559,0.0149,0.062,True,Memish 2014
7,obesity F,0.335,0.3339,-0.0011,-0.003,True,Memish 2014



Joint checks:
== Joint-relationship checks (crude ORs) ==
HTN vs diabetes OR      : 9.48   (input 5.08)
BMI>=25 vs diabetes OR  : 4.54   (input 3.20; age confounding)
Age>=40 vs diabetes OR  : 4.63   (target 4.47)
Diabetes prevalence age>=50: 43.1%  (soft check only)

DATA SAVED SUCCESSFULLY
File: C:\Users\Admin\OneDrive\Documents\Diabetes_Analysis\data\output\synthetic_saudi_diabetes_central.csv
Size: 1118.28 KB
These records are synthetic, not real patients.


## Diagnose age-range Failure

In [21]:
print("Minimum age:", df["age"].min())
print("Maximum age:", df["age"].max())

print("\nAge values below 18:")
display(df.loc[df["age"] < 18, ["id", "sex", "age", "age_group"]].head(10))

print("\nAge values above 85:")
display(df.loc[df["age"] > 85, ["id", "sex", "age", "age_group"]].head(10))

print("\nAge data type:", df["age"].dtype)

Minimum age: 15
Maximum age: 85

Age values below 18:


,id,sex,age,age_group
16,SYN000016,F,16,15-24
17,SYN000017,M,15,15-24
27,SYN000027,M,15,15-24
51,SYN000051,M,15,15-24
62,SYN000062,F,16,15-24
78,SYN000078,F,16,15-24
84,SYN000084,M,16,15-24
89,SYN000089,F,16,15-24
149,SYN000149,M,15,15-24
153,SYN000153,F,17,15-24



Age values above 85:


,id,sex,age,age_group



Age data type: int64
